# Unified Model Comparison: 5 Architectures + XAI

| # | Model | Type | XAI Method |
|---|-------|------|------------|
| 1 | 3D-CNN | 3D Convolution | Gradient Saliency |
| 2 | SSViT v2 | Custom ViT | Attention Rollout |
| 3 | Original ViT | Standard ViT | Attention Rollout |
| 4 | Original ResNet | Standard 2D ResNet | Gradient Saliency |
| 5 | HSI-ResNet | Custom Dual-Path | Gradient Saliency |

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os, seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print('TF:', tf.__version__, '| GPUs:', len(tf.config.list_physical_devices('GPU')))

NUM_CLASSES, BATCH_SIZE, EPOCHS = 3, 64, 30
class_names = ['Fresh', 'Disease', 'Black']

try:
    X = np.load(os.path.join('output_dataset_patches_3d', 'X.npy'))
    y = np.load(os.path.join('output_dataset_patches_3d', 'y.npy'))
except:
    X = np.load('/kaggle/input/tulsi-hyperspectral-patches/X.npy')
    y = np.load('/kaggle/input/tulsi-hyperspectral-patches/y.npy')

N, H, W, C = X.shape
print(f'Data: {X.shape}')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.reshape(-1, C)).reshape(N, H, W, C)
y_cat = tf.keras.utils.to_categorical(y, NUM_CLASSES)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_cat, test_size=0.2, stratify=y, random_state=42)
class_weights = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y), y=y)))
print(f'Train: {X_train.shape}, Test: {X_test.shape}, Weights: {class_weights}')

# 3D format for 3D-CNN
X_3d_train = np.transpose(X_train[..., np.newaxis], (0, 3, 1, 2, 4))
X_3d_test = np.transpose(X_test[..., np.newaxis], (0, 3, 1, 2, 4))
print(f'3D format: {X_3d_train.shape}')

# Store results
results = {}

In [ ]:
# ===== SHARED HELPERS =====

def get_lr_schedule():
    steps = (len(X_train) * 9 // 10) // BATCH_SIZE * EPOCHS
    return tf.keras.optimizers.schedules.CosineDecay(0.001, steps, alpha=0.01)

def get_callbacks(name):
    os.makedirs(f'ckpt_{name}', exist_ok=True)
    return [
        tf.keras.callbacks.ModelCheckpoint(f'ckpt_{name}/best.keras', save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
        tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True, monitor='val_accuracy')
    ]

def train_and_eval(model, name, X_tr, X_te, y_tr, y_te, cw=None):
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=get_lr_schedule(), weight_decay=0.0001),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(X_tr, y_tr, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.1,
                     callbacks=get_callbacks(name), class_weight=cw)
    loss, acc = model.evaluate(X_te, y_te)
    y_prob = model.predict(X_te)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.argmax(y_te, axis=1)
    results[name] = {'acc': acc, 'hist': hist, 'y_prob': y_prob, 'y_pred': y_pred, 'y_true': y_true}
    print(f'\n{"="*50}\n{name} ACCURACY: {acc*100:.2f}%\n{"="*50}')
    print(classification_report(y_true, y_pred, target_names=class_names))
    plt.figure(figsize=(8,6))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{name} - Confusion Matrix ({acc*100:.2f}%)')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.savefig(f'cm_{name}.png', dpi=150); plt.show()
    return model

print('Helpers ready.')

---
# Model 1: 3D-CNN

In [ ]:
def create_3dcnn():
    inp = layers.Input(shape=(168, 5, 5, 1))
    x = layers.Conv3D(8, (7,3,3), activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x); x = layers.MaxPool3D((2,1,1))(x)
    x = layers.Conv3D(16, (5,3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPool3D((2,1,1))(x)
    x = layers.Conv3D(32, (3,3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPool3D((2,1,1))(x)
    x = layers.Conv3D(64, (3,3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x); x = layers.Dropout(0.3)(x)
    return Model(inp, layers.Dense(NUM_CLASSES, activation='softmax')(x), name='3D_CNN')

model_3dcnn = train_and_eval(create_3dcnn(), '3D-CNN', X_3d_train, X_3d_test, y_train, y_test)

---
# Model 2: SSViT v2 (Custom Spectral-Spatial Vision Transformer)

In [ ]:
PROJ_DIM, N_HEADS, N_LAYERS, MLP_DIM = 128, 8, 4, 256

class SSViT_v2(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.reshape_layer = layers.Reshape((25, 168))
        self.projection = layers.Dense(PROJ_DIM)
        self.pos_emb = self.add_weight(name='pos_emb', shape=(1, 25, PROJ_DIM), initializer='random_normal', trainable=True)
        self.blocks = []
        for _ in range(N_LAYERS):
            self.blocks.append({
                'ln1': layers.LayerNormalization(epsilon=1e-6),
                'mha': layers.MultiHeadAttention(num_heads=N_HEADS, key_dim=PROJ_DIM, dropout=0.1),
                'ln2': layers.LayerNormalization(epsilon=1e-6),
                'mlp': tf.keras.Sequential([layers.Dense(MLP_DIM, activation='gelu'), layers.Dropout(0.3),
                                            layers.Dense(PROJ_DIM), layers.Dropout(0.3)]),
                'add1': layers.Add(), 'add2': layers.Add()
            })
        self.final_ln = layers.LayerNormalization(epsilon=1e-6)
        self.pool = layers.GlobalAveragePooling1D()
        self.drop = layers.Dropout(0.3)
        self.fc1 = layers.Dense(MLP_DIM, activation='gelu')
        self.fc2 = layers.Dense(NUM_CLASSES, activation='softmax')

    def call(self, x, return_attention=False, training=False):
        x = self.projection(self.reshape_layer(x)) + self.pos_emb
        attn_scores = []
        for b in self.blocks:
            x1 = b['ln1'](x)
            ao, sc = b['mha'](x1, x1, return_attention_scores=True, training=training)
            if return_attention: attn_scores.append(sc)
            x2 = b['add1']([x, ao])
            x = b['add2']([x2, b['mlp'](b['ln2'](x2), training=training)])
        out = self.fc2(self.fc1(self.drop(self.pool(self.final_ln(x)), training=training)))
        return (out, attn_scores) if return_attention else out

ssvit = SSViT_v2(); _ = ssvit(tf.random.normal((1,5,5,168)))
model_ssvit = train_and_eval(ssvit, 'SSViT-v2', X_train, X_test, y_train, y_test, cw=class_weights)

---
# Model 3: Original ViT (with CLS Token)

In [ ]:
class OriginalViT(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.reshape_layer = layers.Reshape((25, 168))
        self.projection = layers.Dense(PROJ_DIM)
        self.cls_token = self.add_weight(name='cls', shape=(1, 1, PROJ_DIM), initializer='random_normal', trainable=True)
        self.pos_emb = self.add_weight(name='pos_emb', shape=(1, 26, PROJ_DIM), initializer='random_normal', trainable=True)
        self.blocks = []
        for _ in range(N_LAYERS):
            self.blocks.append({
                'ln1': layers.LayerNormalization(epsilon=1e-6),
                'mha': layers.MultiHeadAttention(num_heads=N_HEADS, key_dim=PROJ_DIM, dropout=0.1),
                'ln2': layers.LayerNormalization(epsilon=1e-6),
                'mlp': tf.keras.Sequential([layers.Dense(MLP_DIM, activation='gelu'), layers.Dropout(0.3),
                                            layers.Dense(PROJ_DIM), layers.Dropout(0.3)]),
                'add1': layers.Add(), 'add2': layers.Add()
            })
        self.final_ln = layers.LayerNormalization(epsilon=1e-6)
        self.head = tf.keras.Sequential([layers.Dense(MLP_DIM, activation='gelu'), layers.Dropout(0.3),
                                         layers.Dense(NUM_CLASSES, activation='softmax')])

    def call(self, x, return_attention=False, training=False):
        B = tf.shape(x)[0]
        x = self.projection(self.reshape_layer(x))
        cls = tf.broadcast_to(self.cls_token, [B, 1, PROJ_DIM])
        x = tf.concat([cls, x], axis=1) + self.pos_emb
        attn_scores = []
        for b in self.blocks:
            x1 = b['ln1'](x)
            ao, sc = b['mha'](x1, x1, return_attention_scores=True, training=training)
            if return_attention: attn_scores.append(sc)
            x2 = b['add1']([x, ao])
            x = b['add2']([x2, b['mlp'](b['ln2'](x2), training=training)])
        out = self.head(self.final_ln(x)[:, 0])  # CLS token only
        return (out, attn_scores) if return_attention else out

ovit = OriginalViT(); _ = ovit(tf.random.normal((1,5,5,168)))
model_ovit = train_and_eval(ovit, 'Original-ViT', X_train, X_test, y_train, y_test)

---
# Model 4: Original ResNet (Adapted)

In [ ]:
def res_block(x, f, stride=1):
    s = x
    x = layers.Conv2D(f, 3, strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x)
    if s.shape[-1] != f or stride != 1:
        s = layers.Conv2D(f, 1, strides=stride, padding='same')(s)
        s = layers.BatchNormalization()(s)
    return layers.ReLU()(layers.Add()([x, s]))

def create_orig_resnet():
    inp = layers.Input(shape=(5, 5, 168))
    x = layers.Conv2D(64,(1,1), activation='relu')(inp); x = layers.BatchNormalization()(x)
    x = layers.Conv2D(3,(1,1), activation='relu')(x); x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2,2), interpolation='bilinear')(x)
    x = layers.UpSampling2D((2,2), interpolation='bilinear')(x)
    x = layers.Conv2D(64,(3,3), padding='same')(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.UpSampling2D((2,2), interpolation='bilinear')(x)
    x = res_block(x, 64); x = res_block(x, 64)
    x = res_block(x, 128, stride=2); x = res_block(x, 128)
    x = res_block(x, 256, stride=2); x = res_block(x, 256)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.3)(x)
    return Model(inp, layers.Dense(NUM_CLASSES, activation='softmax')(x), name='Original_ResNet')

model_oresnet = train_and_eval(create_orig_resnet(), 'Original-ResNet', X_train, X_test, y_train, y_test)

---
# Model 5: HSI-ResNet (Custom — Spectral Attention + Dual Path)

In [ ]:
def se_block(x, r=8):
    ch = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(ch//r, activation='relu')(se)
    se = layers.Dense(ch, activation='sigmoid')(se)
    return layers.Multiply()([x, layers.Reshape((1,1,ch))(se)])

def spec_res(x, f):
    s = x
    x = layers.Conv1D(f, 7, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv1D(f, 5, padding='same')(x); x = layers.BatchNormalization()(x)
    if s.shape[-1] != f:
        s = layers.Conv1D(f, 1)(s); s = layers.BatchNormalization()(s)
    return layers.ReLU()(layers.Add()([x, s]))

def spat_res(x, f):
    s = x
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x)
    if s.shape[-1] != f:
        s = layers.Conv2D(f, 1)(s); s = layers.BatchNormalization()(s)
    return layers.ReLU()(layers.Add()([x, s]))

def create_hsi_resnet():
    inp = layers.Input(shape=(5, 5, 168))
    xa = se_block(inp, r=8)
    # Path 1: Spectral
    s1 = layers.Reshape((168, 1))(layers.Lambda(lambda x: x[:, 2, 2, :])(xa))
    s1 = spec_res(s1, 32); s1 = layers.MaxPool1D(2)(s1)
    s1 = spec_res(s1, 64); s1 = layers.MaxPool1D(2)(s1)
    s1 = spec_res(s1, 128); s1 = layers.GlobalAveragePooling1D()(s1)
    # Path 2: Spatial
    s2 = layers.Conv2D(64, (1,1), activation='relu')(xa); s2 = layers.BatchNormalization()(s2)
    s2 = spat_res(s2, 64); s2 = spat_res(s2, 128); s2 = spat_res(s2, 128)
    s2 = layers.GlobalAveragePooling2D()(s2)
    # Fusion
    x = layers.Concatenate()([s1, s2])
    x = layers.Dense(256, activation='relu')(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.3)(x)
    return Model(inp, layers.Dense(NUM_CLASSES, activation='softmax')(x), name='HSI_ResNet')

model_hresnet = train_and_eval(create_hsi_resnet(), 'HSI-ResNet', X_train, X_test, y_train, y_test)

---
# Explainable AI (XAI) for All 5 Models

In [ ]:
# ===== XAI FUNCTIONS =====

def gradient_saliency(model, sample):
    s = tf.cast(tf.constant(sample), tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(s)
        p = model(s, training=False)
        out = p[0, tf.argmax(p[0])]
    g = tape.gradient(out, s).numpy()[0]
    if g.ndim == 4:  # 3D input (168,5,5,1)
        return np.mean(np.abs(g[:,:,:,0]), axis=0).flatten()
    return np.mean(np.abs(g), axis=-1).flatten()

def attention_rollout(model, sample):
    _, attn = model(sample, return_attention=True, training=False)
    n = 25
    roll = np.eye(n)
    for a in attn:
        avg = tf.reduce_mean(a, axis=1).numpy()[0]
        if avg.shape[0] > n:  # Has CLS token
            avg = avg[1:, 1:]  # Remove CLS row/col
        avg = avg + np.eye(n)
        avg = avg / avg.sum(axis=-1, keepdims=True)
        roll = np.matmul(avg, roll)
    return roll.mean(axis=0)

def xai_6panel(idx, model, mask, y_t, y_p, y_prob, sample_2d, name):
    label, pred = y_t[idx], y_p[idx]
    conf = y_prob[idx][pred] * 100
    mn = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    m2d = mn.reshape(5, 5)
    mi = np.argmax(mask)
    mr, mc = mi // 5, mi % 5
    rgb = np.stack([sample_2d[0,:,:,100], sample_2d[0,:,:,60], sample_2d[0,:,:,20]], axis=-1)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    ms = sample_2d[0].mean(axis=(0,1))

    fig = plt.figure(figsize=(16, 8))
    ax1 = fig.add_subplot(2,3,1); ax1.imshow(rgb, interpolation='nearest')
    ax1.set_title(f'Patch\nTrue: {class_names[label]} | Pred: {class_names[pred]}'); ax1.axis('off')

    ax2 = fig.add_subplot(2,3,2)
    im = ax2.imshow(m2d, cmap='hot', interpolation='nearest', vmin=0, vmax=1)
    ax2.scatter(mc, mr, marker='x', color='cyan', s=200, linewidths=3)
    ax2.set_title(f'Saliency Heatmap\nKey: Row {mr+1}, Col {mc+1}'); ax2.axis('off')
    plt.colorbar(im, ax=ax2, fraction=0.046)

    ax3 = fig.add_subplot(2,3,3)
    ax3.bar(range(25), mn, color=plt.cm.hot(mn))
    ax3.set_xlabel('Pixel (1-25)'); ax3.set_ylabel('Importance'); ax3.set_title('Pixel Ranking')

    ax4 = fig.add_subplot(2,3,4)
    ax4.plot(ms, color='darkgreen', lw=1.5); ax4.fill_between(range(len(ms)), ms, alpha=0.3, color='green')
    ax4.set_xlabel('Band (0-167)'); ax4.set_title('Mean Spectral Signature'); ax4.grid(True, alpha=0.3)

    ax5 = fig.add_subplot(2,3,5); ax5.axis('off')
    txt = (f'=== {name} XAI ===\n\nTrue: {class_names[label]}\nPred: {class_names[pred]}\n'
           f'Conf: {conf:.1f}%\n\nKey Pixel: ({mr+1},{mc+1})\n'
           f'Top5: {np.argsort(mask)[-5:][::-1]+1}')
    ax5.text(0.1, 0.5, txt, fontsize=11, family='monospace', va='center',
             transform=ax5.transAxes, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax6 = fig.add_subplot(2,3,6)
    bars = ax6.barh(class_names, y_prob[idx]*100, color=['green','orange','black'])
    ax6.set_xlabel('Probability (%)'); ax6.set_title('Confidence'); ax6.set_xlim(0,100)
    for b, p in zip(bars, y_prob[idx]):
        ax6.text(p*100+2, b.get_y()+b.get_height()/2, f'{p*100:.1f}%', va='center')

    plt.suptitle(f'{name} — Sample {idx}', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.savefig(f'xai_{name.replace(" ","_").lower()}_{idx}.png', dpi=150); plt.show()
    print(f'{name} | Sample {idx} | True: {class_names[label]} | Pred: {class_names[pred]} ({conf:.1f}%) | {"CORRECT" if label==pred else "WRONG"}')

print('XAI functions ready.')

In [ ]:
# ===== RUN XAI FOR ALL 5 MODELS =====
samples = [0, 10, 20]
r = results

# 1. 3D-CNN (Gradient Saliency on 3D input)
print('\n' + '#'*60 + '\n XAI: 3D-CNN\n' + '#'*60)
for i in samples:
    mask = gradient_saliency(model_3dcnn, X_3d_test[i:i+1])
    xai_6panel(i, model_3dcnn, mask, r['3D-CNN']['y_true'], r['3D-CNN']['y_pred'], r['3D-CNN']['y_prob'], X_test[i:i+1], '3D-CNN')

# 2. SSViT v2 (Attention Rollout)
print('\n' + '#'*60 + '\n XAI: SSViT v2\n' + '#'*60)
for i in samples:
    mask = attention_rollout(model_ssvit, X_test[i:i+1])
    xai_6panel(i, model_ssvit, mask, r['SSViT-v2']['y_true'], r['SSViT-v2']['y_pred'], r['SSViT-v2']['y_prob'], X_test[i:i+1], 'SSViT-v2')

# 3. Original ViT (Attention Rollout)
print('\n' + '#'*60 + '\n XAI: Original ViT\n' + '#'*60)
for i in samples:
    mask = attention_rollout(model_ovit, X_test[i:i+1])
    xai_6panel(i, model_ovit, mask, r['Original-ViT']['y_true'], r['Original-ViT']['y_pred'], r['Original-ViT']['y_prob'], X_test[i:i+1], 'Original-ViT')

# 4. Original ResNet (Gradient Saliency)
print('\n' + '#'*60 + '\n XAI: Original ResNet\n' + '#'*60)
for i in samples:
    mask = gradient_saliency(model_oresnet, X_test[i:i+1])
    xai_6panel(i, model_oresnet, mask, r['Original-ResNet']['y_true'], r['Original-ResNet']['y_pred'], r['Original-ResNet']['y_prob'], X_test[i:i+1], 'Original-ResNet')

# 5. HSI-ResNet (Gradient Saliency)
print('\n' + '#'*60 + '\n XAI: HSI-ResNet\n' + '#'*60)
for i in samples:
    mask = gradient_saliency(model_hresnet, X_test[i:i+1])
    xai_6panel(i, model_hresnet, mask, r['HSI-ResNet']['y_true'], r['HSI-ResNet']['y_pred'], r['HSI-ResNet']['y_prob'], X_test[i:i+1], 'HSI-ResNet')

In [ ]:
# ===== SIDE-BY-SIDE XAI COMPARISON (All 5 at once) =====

for idx in [0, 10, 20]:
    masks = [
        gradient_saliency(model_3dcnn, X_3d_test[idx:idx+1]),
        attention_rollout(model_ssvit, X_test[idx:idx+1]),
        attention_rollout(model_ovit, X_test[idx:idx+1]),
        gradient_saliency(model_oresnet, X_test[idx:idx+1]),
        gradient_saliency(model_hresnet, X_test[idx:idx+1]),
    ]
    names = ['3D-CNN', 'SSViT-v2', 'Original-ViT', 'Orig-ResNet', 'HSI-ResNet']
    
    fig, axes = plt.subplots(1, 6, figsize=(24, 4))
    rgb = np.stack([X_test[idx,:,:,100], X_test[idx,:,:,60], X_test[idx,:,:,20]], axis=-1)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    axes[0].imshow(rgb, interpolation='nearest')
    axes[0].set_title(f'Sample #{idx}\nTrue: {class_names[r["3D-CNN"]["y_true"][idx]]}')
    axes[0].axis('off')
    
    for j, (m, n) in enumerate(zip(masks, names)):
        mn = (m - m.min()) / (m.max() - m.min() + 1e-8)
        im = axes[j+1].imshow(mn.reshape(5,5), cmap='hot', interpolation='nearest', vmin=0, vmax=1)
        pred = r[n if n != 'Orig-ResNet' else 'Original-ResNet']['y_pred'][idx]
        acc = r[n if n != 'Orig-ResNet' else 'Original-ResNet']['acc']
        axes[j+1].set_title(f'{n}\nPred: {class_names[pred]} ({acc*100:.1f}%)')
        axes[j+1].axis('off')
    
    plt.suptitle(f'XAI Comparison — All 5 Models — Sample {idx}', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.savefig(f'xai_all5_compare_{idx}.png', dpi=150); plt.show()

---
# Final Comparison

In [ ]:
# ===== TRAINING CURVES =====
fig, axes = plt.subplots(2, 5, figsize=(25, 8))
for i, (n, r_data) in enumerate(results.items()):
    axes[0,i].plot(r_data['hist'].history['accuracy'], label='Train')
    axes[0,i].plot(r_data['hist'].history['val_accuracy'], label='Val')
    axes[0,i].set_title(f'{n}'); axes[0,i].legend(); axes[0,i].grid(True)
    axes[1,i].plot(r_data['hist'].history['loss'], label='Train')
    axes[1,i].plot(r_data['hist'].history['val_loss'], label='Val')
    axes[1,i].set_title(f'{n} Loss'); axes[1,i].legend(); axes[1,i].grid(True)
plt.suptitle('Training Curves — All 5 Models', fontsize=14)
plt.tight_layout(); plt.savefig('all_training_curves.png', dpi=150); plt.show()

# ===== ACCURACY BAR CHART =====
plt.figure(figsize=(10, 5))
names = list(results.keys())
accs = [results[n]['acc']*100 for n in names]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']
bars = plt.bar(names, accs, color=colors, edgecolor='black')
for b, a in zip(bars, accs):
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{a:.2f}%', ha='center', fontweight='bold')
plt.ylabel('Test Accuracy (%)'); plt.title('Model Accuracy Comparison')
plt.ylim(min(accs)-5, 100); plt.grid(axis='y', alpha=0.3)
plt.savefig('accuracy_comparison.png', dpi=150); plt.show()

# ===== FINAL TABLE =====
print('\n' + '='*80)
print('FINAL MODEL COMPARISON — TULSI HYPERSPECTRAL CLASSIFICATION')
print('='*80)
print(f'{"Model":<25} {"Type":<25} {"HSI-Specific":<15} {"Accuracy":<10}')
print('-'*80)
meta = [
    ('3D-CNN', '3D Convolution', 'Partial'),
    ('SSViT-v2', 'Custom ViT', 'YES'),
    ('Original-ViT', 'Standard CLS ViT', 'No'),
    ('Original-ResNet', 'Standard 2D ResNet', 'No'),
    ('HSI-ResNet', 'SE + Dual-Path ResNet', 'YES'),
]
for n, t, h in meta:
    print(f'{n:<25} {t:<25} {h:<15} {results[n]["acc"]*100:.2f}%')
print('='*80)
best = max(results, key=lambda k: results[k]['acc'])
print(f'\nBest Model: {best} ({results[best]["acc"]*100:.2f}%)')